Retreive 20 records including the last 4 columns from the GCS's file generated from ex01.ipynb
Using "gemini-3.6-flash" (or a version it recommends), ask it to summarize the reports.

- Pre-requisite : Generate API keys through [Google AI Studio](https://aistudio.google.com/app/apikey)

In [ ]:
import datetime
import json
import os
from dotenv import load_dotenv
from google import genai
from google.oauth2 import service_account
from google.cloud import storage

In [31]:
from pathlib import Path
load_dotenv(Path("/Users/prashastisrivastava/Class/2026_MSDS692/dsai692_data_acquisition_2026/.env"))

True

In [32]:
gemini_api_key = os.getenv("GEMINI_API_KEY")
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
file_name = f"sf_police_report/{datetime.date.today()}.json"

In [33]:
# def retrieve_data_from_gcs(service_account_key: str,
#                            project_id: str,
#                            bucket_name: str,
#                            file_name: str,
#                            key_list: list
#                            ) -> list:
#     credentials = service_account.Credentials.from_service_account_file(service_account_key)
#     client = storage.Client(project=project_id,
#                             credentials=credentials)
#     bucket = client.bucket(bucket_name)
#     file = bucket.blob(file_name)
#     content = json.loads(file.download_as_string())

#     # TODO : Process content and only return fields within key_list

    

In [34]:
def retrieve_data_from_gcs(service_account_key: str,
                           project_id: str,
                           bucket_name: str,
                           file_name: str,
                           key_list: list
                           ) -> list:
    credentials = service_account.Credentials.from_service_account_file(service_account_key)
    client = storage.Client(project=project_id,
                            credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    content = json.loads(file.download_as_string())

    output = []
    for data in content:
        row = []
        
        for key in key_list:
            row.append(data.get(key, None))
        output.append(row)
    return output

In [35]:
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
data = retrieve_data_from_gcs(service_account_key, 
                              project_id,
                              bucket_name,
                              file_name,
                              key_list)

In [36]:
filtered_data = [row[-4:] for row in data][:20]

In [37]:
import os
from dotenv import load_dotenv, find_dotenv
print("cwd:", os.getcwd())
print("dotenv found at:", repr(find_dotenv()))
load_dotenv(find_dotenv(), override=True)
print([k for k in os.environ if "API" in k or "GEMINI" in k or "GOOGLE" in k])

cwd: /Users/prashastisrivastava/Class/2026_MSDS692/dsai692_data_acquisition_2026/Day4/Skeleton
dotenv found at: '/Users/prashastisrivastava/Class/2026_MSDS692/dsai692_data_acquisition_2026/.env'
['GEMINI_API_KEY', 'DATA_GOV_API_KEY']


In [38]:
# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [39]:
model_name = "gemini-3.6-flash"

In [40]:
prompt_content = f"There has been a police report\
on the following list of [description, lon, lat, district] : {filtered_data} recently.\
Summarize the reports"

In [47]:
# response = # TODO : See the API Document and complete to call Gemini with prompt

response = client.models.generate_content(
    model=model_name,
    contents=prompt_content
)

In [48]:
# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client()
response = client.models.generate_content(
model='gemini-3.6-flash', contents="Explain how AI works in a few words"
)
print(response.text)

AI works by finding **patterns in vast amounts of data** to **predict**, **decide**, or **create** new outputs. 

In short: **Data in → Patterns learned → Smart output out.**


In [42]:
response.text

'Here is a summary of the 20 recent police reports, categorized by police district and type of crime.\n\n---\n\n### **Executive Summary**\n* **Total Incidents Reported:** 20\n* **Most Active Districts:** **Ingleside** (5 incidents) and **Mission** (4 incidents)\n* **Primary Crime Types:** \n  * Property Theft & Burglary (6)\n  * Violent Crimes / Assaults / Robberies (5)\n  * Narcotics / Drug Offenses (2)\n  * Vehicle-Related Incidents (2)\n  * Fraud, Warrants & Code Violations (5)\n\n---\n\n### **Breakdown by Police District**\n\n#### **1. Ingleside (5 reports)**\n* **Theft & Vehicle Crimes:** 1x Theft from Unlocked Vehicle (>$950), 1x Stolen Auto\n* **Violent/Personal Crimes:** 1x Battery\n* **Other:** 1x Violation of Restraining Order, 1x Miscellaneous Investigation\n\n#### **2. Mission (4 reports)**\n* **Theft & Robbery:** 1x Robbery with Force (Street/Public Place), 1x Theft from Unlocked Vehicle (>$950), 1x Theft ($50–$200)\n* **Drugs:** 1x Possession of Narcotics Paraphernalia\n\

In [43]:
# Note: This is an optional to display markdown strings.
from rich.console import Console
from rich.markdown import Markdown

console = Console()
md = Markdown(response.text)
console.print(md)


Here is a summary of the 20 recent police reports, categorized by police district and type of crime.               

-------------------------------------------------------------------------------------------------------------------

Executive Summary                                                                                                  

 • Total Incidents Reported: 20                                                                                    
 • Most Active Districts: Ingleside (5 incidents) and Mission (4 incidents)                                        
 • Primary Crime Types:                                                                                            
    • Property Theft & Burglary (6)                                                                                
    • Violent Crimes / Assaults / Robberies (5)                                                                    
    • Narcotics / Drug Offenses (2)                                                                                
    • Vehicle-Related Incidents (2)                                                                                
    • Fraud, Warrants & Code Violations (5)                                                                        

-------------------------------------------------------------------------------------------------------------------

Breakdown by Police District                                                                                       

1. Ingleside (5 reports)                                                                                           

 • Theft & Vehicle Crimes: 1x Theft from Unlocked Vehicle (>$950), 1x Stolen Auto                                  
 • Violent/Personal Crimes: 1x Battery                                                                             
 • Other: 1x Violation of Restraining Order, 1x Miscellaneous Investigation                                        

2. Mission (4 reports)                                                                                             

 • Theft & Robbery: 1x Robbery with Force (Street/Public Place), 1x Theft from Unlocked Vehicle (>$950), 1x Theft  
   ($50–$200)                                                                                                      
 • Drugs: 1x Possession of Narcotics Paraphernalia                                                                 

3. Bayview (2 reports)                                                                                             

 • Violent Crimes: 1x Aggravated Assault with Force                                                                
 • Fraud: 1x False Personation                                                                                     

4. Northern (2 reports)                                                                                            

 • Theft & Burglary: 1x Robbery with Force, 1x Residential Burglary (Unlawful Entry)                               

5. Richmond (2 reports)                                                                                            

 • Property/Suspicious: 1x Theft from Building (>$950), 1x Suspicious Occurrence                                   

6. Tenderloin (2 reports)                                                                                          

 • Drugs & Code Violations: 1x Heroin Offense, 1x Municipal Police Code Violation                                  

7. Out of SF (2 reports)                                                                                           

 • Warrants & Recovery: 1x Local SF Warrant Arrest, 1x Vehicle Recovered                                           

8. Central (1 report)                                                                                              

 • Violent Crimes: 1x Battery with Serious Injuries                                                                

----------------------------------

In [44]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True